# Multi-Agent Debate
## Agree to Disagree — Multi-Perspective Argument Synthesis
⏱ ~12 min

**Multi-agent debate** is a prompting strategy where independent LLM agents argue opposing positions, exchange structured critiques, and a judge aggregates the strongest reasoning into a final verdict. Research by Du et al. (2023) showed this approach improves factual accuracy and reasoning quality compared to a single model answering alone.

By the end of this workshop you will understand *why* debate improves LLM output, *how* every node in the graph fits together, and *how* to extend the pattern to your own domains.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why debate works and when to use it |
| 2 | **State & Roles** — DebateState, system prompts, the LLM client |
| 3 | **Graph Nodes** — solve\_a, solve\_b, debate, judge |
| 4 | **Wiring the Graph** — LangGraph StateGraph and conditional edges |
| 5 | **Running a Debate** — full execution and result inspection |
| 6 | **Tuning** — round count, temperature, judge prompt engineering |
| 7 | **Parsing & Metrics** — extracting structured verdicts programmatically |
| ★ | **Extensions** — three-way debate, consensus mode, streaming |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the repo `requirements.txt`
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Du, Y., Li, S., Torralba, A., Tenenbaum, J. B., & Mordatch, I. (2023). *Improving Factuality and Reasoning in Language Models through Multiagent Debate.* ICML 2024. https://arxiv.org/abs/2305.14325
>
> Liang, T., He, Z., Jiao, W., Wang, X., Wang, Y., Wang, R., ... & Shi, S. (2023). *Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate.* https://arxiv.org/abs/2305.19118
>
> LangGraph documentation: https://langchain-ai.github.io/langgraph/
>
> LangChain StateGraph API: https://python.langchain.com/docs/langgraph

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or malformed. "
    "Local: add it to .env. Colab: add it in Secrets panel."
)
print(f"OPENAI_API_KEY present: {bool(key)} (starts with sk-)")

---

## Part 1 — Concepts: Why Debate Works

---

### The Problem with Single-Agent Reasoning

A single LLM tends to:
- **Confirm its priors** — the first plausible answer it generates is rarely challenged
- **Hallucinate confidently** — it has no external check on factual claims
- **Miss alternative framings** — it explores one line of reasoning rather than the full space

Debate forces the model out of this attractor. When an agent *knows* a counter-argument is coming, it reasons more carefully from the start.

---

### How Multi-Agent Debate Helps

| Mechanism | What it does |
|-----------|-------------|
| **Independent initialization** | Each agent is seeded with an opposing system prompt, preventing premature convergence |
| **Critique exchange** | Each agent reads the opponent's strongest argument before refining its own |
| **Iterative refinement** | Multiple rounds let both positions sharpen toward the strongest defensible claim |
| **Impartial judge** | A third agent (with no position) selects the winner using only the final arguments |

> Du et al. (2023) found multi-agent debate improved accuracy on math reasoning (GSM8K), factual question answering, and chess move analysis — all with no fine-tuning, only prompting.

---

### Debate Formats Compared

| Format | Agents | Rounds | Judge | Best for |
|--------|--------|--------|-------|----------|
| **Binary debate** (this workshop) | 2 | 1–3 | 1 impartial | Yes/no decisions, A vs B comparisons |
| **Panel debate** | 3–5 | 1–2 | Majority vote or 1 judge | Complex multi-factor decisions |
| **Society of Mind** | N | N | Aggregated | Long-horizon reasoning, research |
| **Self-critique** | 1 | N | Same agent | Simpler tasks, cost-sensitive |

---

### Debate Topology (ASCII)

```
              TOPIC
                |
       +--------+--------+
       |                 |
 +------------+   +------------+
 |  Agent A   |   |  Agent B   |
 |(Affirmative|   | (Negative  |
 | position)  |   |  position) |
 +-----+------+   +------+-----+
       |   critique A->B  |
       |<-----------------|
       |   critique B->A  |
       |----------------->|
       |                  |
       +------ round N ---+
              (repeat)
                |
                v
         +------------+
         |   Judge    |
         |   Agent    |
         +-----+------+
               |
               v
           VERDICT
        WINNER: [A or B]
```

---

### Round Count Trade-offs

| Rounds | Behaviour | LLM calls | When to use |
|--------|-----------|-----------|-------------|
| **1** | Opening arguments only; no refinement | 4 | Quick gut-check; cost-sensitive pipelines |
| **2** | One critique exchange; positions sharpen meaningfully | 8 | Most production use cases |
| **3** | Second refinement; diminishing returns begin | 12 | High-stakes decisions |
| **4+** | Positions often restabilize; marginal gain | 16+ | Research / benchmarking only |

---

## Part 2 — State and Roles

---

### DebateState — the shared memory

LangGraph passes a single `state` dict through every node. Each node reads what it needs and returns only the keys it updates — no node sees implementation details of any other node.

```
DebateState
+-- topic       str   the proposition being debated
+-- position_a  str   Agent A's current argument
+-- position_b  str   Agent B's current argument
+-- critique_a  str   B's critique of A (A reads this to refine)
+-- critique_b  str   A's critique of B (B reads this to refine)
+-- round       int   how many critique exchanges have occurred
+-- verdict     str   judge's final verdict (set only in last node)
```

### System Prompt Design

The system prompt is the **only** thing that separates Agent A from Agent B — same LLM, same temperature, different role. You don't need different models or fine-tuning to get genuinely different perspectives.

| Agent | System prompt strategy | What it prevents |
|-------|----------------------|------------------|
| **Agent A** | Locked into affirmative position; instructed to steelman then reinforce | Switching sides to agree with B |
| **Agent B** | Locked into opposing position; same steelman instruction | Switching sides to agree with A |
| **Judge** | No position; instructed to select stronger *argument*, not preferred conclusion | Bias toward one side's framing |

In [ ]:
# ─── 2-A: Define DebateState and system prompts ───────────────────────────────
from typing import TypedDict


class DebateState(TypedDict):
    topic: str
    position_a: str
    position_b: str
    critique_a: str   # B's critique of A — A must read and respond to this
    critique_b: str   # A's critique of B — B must read and respond to this
    round: int
    verdict: str


MAX_ROUNDS = 2
TOPIC = "Is specialization or generalization better for a software engineer's career?"

SOLVER_A_SYSTEM = (
    "You are a career expert arguing FOR SPECIALIZATION in software engineering. "
    "Make compelling, evidence-based arguments in 3-4 sentences. "
    "If given a critique, acknowledge its strongest point then reinforce your position."
)

SOLVER_B_SYSTEM = (
    "You are a career expert arguing FOR GENERALIZATION (being a full-stack or T-shaped engineer). "
    "Make compelling, evidence-based arguments in 3-4 sentences. "
    "If given a critique, acknowledge its strongest point then reinforce your position."
)

JUDGE_SYSTEM = (
    "You are an impartial judge evaluating a structured debate. "
    "Read both final positions and select the stronger argument. "
    "Explain your reasoning in 2-3 sentences, then end with exactly: "
    "WINNER: [SPECIALIZATION or GENERALIZATION]"
)

print("State schema:")
for key, typ in DebateState.__annotations__.items():
    print(f"  {key}: {typ.__name__}")

In [ ]:
# ─── 2-B: Create the LLM client ───────────────────────────────────────────────
# We use a single shared client. Temperature=0.7 adds enough variance
# that the two agents don't collapse to identical phrasing.
# For reproducible benchmarking, set temperature=0.

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print(f"Model: {llm.model_name}")
print(f"Temperature: {llm.temperature}")
print()
print("Both agents share this client — the system prompt is their only difference.")

In [ ]:
# ─── 2-C: Inspect a single raw agent call ─────────────────────────────────────
# Before building the graph, see what one agent call looks like in isolation.

from langchain_core.messages import HumanMessage, SystemMessage

opening_prompt = f"Topic: {TOPIC}\n\nPresent your opening argument FOR SPECIALIZATION:"

response = llm.invoke([
    SystemMessage(content=SOLVER_A_SYSTEM),
    HumanMessage(content=opening_prompt),
])

print("Agent A opening argument (raw call):")
print(response.content)
print()
usage = response.usage_metadata or {}
print(f"Token usage — prompt: {usage.get('input_tokens', 'n/a')}, "
      f"completion: {usage.get('output_tokens', 'n/a')}")

---

## Part 3 — Graph Nodes

---

The debate graph has **four nodes**. Each is a plain Python function that accepts `DebateState` and returns a dict containing only the keys it modifies.

```
solve_a  ->  produces position_a
solve_b  ->  produces position_b
debate   ->  produces critique_a, critique_b, round++
judge    ->  produces verdict
```

The `debate` node runs both critique calls in the same function. This is a design choice — you could split them into separate nodes if you needed to parallelize (LangGraph supports parallel branches via `Send`).

### Node Input/Output Map

| Node | Reads from state | Writes to state |
|------|-----------------|----------------|
| `solve_a` | `topic`, `position_a`, `critique_a` | `position_a` |
| `solve_b` | `topic`, `position_b`, `critique_b` | `position_b` |
| `debate` | `topic`, `position_a`, `position_b`, `round` | `critique_a`, `critique_b`, `round` |
| `judge` | `topic`, `position_a`, `position_b` | `verdict` |

In [ ]:
# ─── 3-A: solve_a — Specialization advocate ───────────────────────────────────
# First call: no critique yet — present opening argument.
# Subsequent calls: read B's critique and refine the position.

def solve_a(state: DebateState) -> dict:
    if state.get("critique_a"):
        # Refinement round: B has critiqued A, so A defends and sharpens
        content = (
            f"Topic: {state['topic']}\n\n"
            f"Your current position: {state['position_a']}\n\n"
            f"Opponent's critique of your argument: {state['critique_a']}\n\n"
            "Refine your argument, addressing the critique:"
        )
    else:
        # Opening round — no prior context
        content = (
            f"Topic: {state['topic']}\n\n"
            "Present your opening argument FOR SPECIALIZATION:"
        )
    result = llm.invoke([
        SystemMessage(content=SOLVER_A_SYSTEM),
        HumanMessage(content=content),
    ])
    return {"position_a": result.content}


# Smoke test — run solve_a in isolation before wiring it into the graph
test_state: DebateState = {
    "topic": TOPIC, "position_a": "", "position_b": "",
    "critique_a": "", "critique_b": "", "round": 0, "verdict": ""
}
result_a = solve_a(test_state)
print("solve_a output (opening round):")
print(result_a["position_a"])

In [ ]:
# ─── 3-B: solve_b — Generalization advocate ───────────────────────────────────
# Mirror of solve_a but locked into the opposing position.

def solve_b(state: DebateState) -> dict:
    if state.get("critique_b"):
        content = (
            f"Topic: {state['topic']}\n\n"
            f"Your current position: {state['position_b']}\n\n"
            f"Opponent's critique of your argument: {state['critique_b']}\n\n"
            "Refine your argument, addressing the critique:"
        )
    else:
        content = (
            f"Topic: {state['topic']}\n\n"
            "Present your opening argument FOR GENERALIZATION:"
        )
    result = llm.invoke([
        SystemMessage(content=SOLVER_B_SYSTEM),
        HumanMessage(content=content),
    ])
    return {"position_b": result.content}


# Smoke test
test_state_b = {**test_state, "position_a": result_a["position_a"]}
result_b = solve_b(test_state_b)
print("solve_b output (opening round):")
print(result_b["position_b"])

In [ ]:
# ─── 3-C: debate and judge nodes ──────────────────────────────────────────────
# debate: each agent critiques the opponent's current position.
#         Both calls happen sequentially inside the same node.
# judge:  impartial — reads final positions only; no prior conversation.

def debate(state: DebateState) -> dict:
    # A critiques B's position (A reads position_b and produces a critique)
    crit_a_content = (
        f"Topic: {state['topic']}\n\n"
        f"Opponent argues: {state['position_b']}\n\n"
        "Write a 2-sentence critique of the opponent's argument:"
    )
    # B critiques A's position
    crit_b_content = (
        f"Topic: {state['topic']}\n\n"
        f"Opponent argues: {state['position_a']}\n\n"
        "Write a 2-sentence critique of the opponent's argument:"
    )
    critique_a = llm.invoke([SystemMessage(content=SOLVER_A_SYSTEM), HumanMessage(content=crit_a_content)])
    critique_b = llm.invoke([SystemMessage(content=SOLVER_B_SYSTEM), HumanMessage(content=crit_b_content)])
    return {
        "critique_a": critique_a.content,   # B->A: what A must respond to next round
        "critique_b": critique_b.content,   # A->B: what B must respond to next round
        "round": state.get("round", 0) + 1,
    }


def judge(state: DebateState) -> dict:
    content = (
        f"Topic: {state['topic']}\n\n"
        f"POSITION A (Specialization):\n{state['position_a']}\n\n"
        f"POSITION B (Generalization):\n{state['position_b']}"
    )
    result = llm.invoke([
        SystemMessage(content=JUDGE_SYSTEM),
        HumanMessage(content=content),
    ])
    return {"verdict": result.content}


print("All four node functions defined: solve_a, solve_b, debate, judge")
print()
print("  solve_a  -> writes position_a")
print("  solve_b  -> writes position_b")
print("  debate   -> writes critique_a, critique_b, round")
print("  judge    -> writes verdict")

---

## Part 4 — Wiring the Graph

---

LangGraph's `StateGraph` manages execution order. The key concept here is the **conditional edge**: after each `debate` node, the graph checks `round` against `MAX_ROUNDS` to decide whether to loop back for another round or hand off to the judge.

```
START
  |
  v
solve_a --> solve_b --> debate
                          |
            round < MAX <-+
                          |
            round >= MAX ---> judge --> END
```

### Conditional Edge Pattern

```python
def should_continue(state) -> str:
    return "solve_a" if state["round"] < MAX_ROUNDS else "judge"

graph.add_conditional_edges(
    "debate",           # source node
    should_continue,    # routing function — returns a string key
    {"solve_a": "solve_a", "judge": "judge"}  # string key -> node name
)
```

This is LangGraph's idiomatic loop pattern: a routing function returns a string that maps to the next node. The dict argument makes the mapping explicit and type-safe.

In [ ]:
# ─── 4-A: Build and compile the debate graph ──────────────────────────────────
from langgraph.graph import END, StateGraph


def should_continue(state: DebateState) -> str:
    """Route back to solve_a for another round, or to judge when done."""
    return "solve_a" if state.get("round", 0) < MAX_ROUNDS else "judge"


def create_debate_graph() -> StateGraph:
    graph = StateGraph(DebateState)

    # Register nodes
    graph.add_node("solve_a", solve_a)
    graph.add_node("solve_b", solve_b)
    graph.add_node("debate", debate)
    graph.add_node("judge", judge)

    # Fixed edges — always follow this path
    graph.set_entry_point("solve_a")
    graph.add_edge("solve_a", "solve_b")
    graph.add_edge("solve_b", "debate")

    # Conditional edge — loops back or exits to judge
    graph.add_conditional_edges(
        "debate",
        should_continue,
        {"solve_a": "solve_a", "judge": "judge"},
    )

    graph.add_edge("judge", END)
    return graph.compile()


app = create_debate_graph()
print("Debate graph compiled.")
print(f"Nodes: {list(app.get_graph().nodes.keys())}")
print(f"MAX_ROUNDS: {MAX_ROUNDS}")
llm_calls = 2 + MAX_ROUNDS * 4
print(f"LLM calls per full run: {llm_calls}  (2 openings + {MAX_ROUNDS} rounds x 4 calls each)")

In [ ]:
# ─── 4-B: Inspect graph structure before committing API spend ─────────────────
# The graph object exposes its topology — verify the wiring first.

g = app.get_graph()

print("Nodes in graph:")
for node_id in g.nodes:
    print(f"  {node_id}")

print()
print("Edges:")
for edge in g.edges:
    print(f"  {edge[0]} -> {edge[1]}")

print()
print("Execution trace for MAX_ROUNDS=2:")
trace = [
    "solve_a (opening argument for Specialization)",
    "solve_b (opening argument for Generalization)",
    "debate  (round 1: mutual critique exchange)",
    "solve_a (refinement — addresses B's critique)",
    "solve_b (refinement — addresses A's critique)",
    "debate  (round 2: second critique exchange)",
    "judge   (reads final positions, emits WINNER)",
]
for i, step in enumerate(trace, 1):
    print(f"  {i}. {step}")

---

## Part 5 — Running a Full Debate

---

Now we invoke the compiled graph with the initial state. The graph runs all nodes automatically in the correct order, passing the updated state dict between them.

**Expected output per round:**
- Round 1: both agents present opening arguments, then exchange critiques
- Round 2: both agents refine their positions in light of the critique, then exchange again
- Judge: reads only the *final* positions — not the full history

In [ ]:
# ─── 5-A: Run the debate ──────────────────────────────────────────────────────

initial_state: DebateState = {
    "topic": TOPIC,
    "position_a": "",
    "position_b": "",
    "critique_a": "",
    "critique_b": "",
    "round": 0,
    "verdict": "",
}

print(f"TOPIC: {TOPIC}")
print("=" * 60)
print("Running debate...")

result = app.invoke(initial_state)

print(f"Completed {result['round']} round(s).")

In [ ]:
# ─── 5-B: Inspect the full result ─────────────────────────────────────────────

divider = "-" * 60

print("FINAL POSITION A — Specialization")
print(divider)
print(result["position_a"])

print()
print("FINAL POSITION B — Generalization")
print(divider)
print(result["position_b"])

print()
print("JUDGE'S VERDICT")
print(divider)
print(result["verdict"])

In [ ]:
# ─── 5-C: Inspect the critique exchange ───────────────────────────────────────
# The final critiques show what each agent identified as the opponent's
# weakest point — useful for understanding why positions evolved.

print("LAST CRITIQUE: B's critique of A (what A had to defend against)")
print(divider)
print(result["critique_a"])

print()
print("LAST CRITIQUE: A's critique of B (what B had to defend against)")
print(divider)
print(result["critique_b"])

---

## Part 6 — Tuning the Debate

---

Three knobs have the biggest impact on debate quality:

| Parameter | Low value | High value | Recommendation |
|-----------|-----------|------------|----------------|
| `MAX_ROUNDS` | Faster, cheaper; positions stay closer to opening | Deeper refinement; higher cost; risk of stagnation after round 3 | 2 for production |
| `temperature` | Deterministic; positions may converge | Creative; more divergent; risk of drift | 0.7 for agents, 0 for judge |
| Critique length | Short: fast, punchy | Long: thorough; may overwhelm the responding agent | 2-3 sentences |

In [ ]:
# ─── 6-A: Compare round counts on a fresh topic ───────────────────────────────
# Run the same topic with MAX_ROUNDS=1 and MAX_ROUNDS=2,
# then compare position length and the judge's confidence.

def run_debate(topic: str, max_rounds: int) -> dict:
    """Build and run a fresh graph with the given round limit."""
    def _should_continue(state: DebateState) -> str:
        return "solve_a" if state.get("round", 0) < max_rounds else "judge"

    g = StateGraph(DebateState)
    g.add_node("solve_a", solve_a)
    g.add_node("solve_b", solve_b)
    g.add_node("debate", debate)
    g.add_node("judge", judge)
    g.set_entry_point("solve_a")
    g.add_edge("solve_a", "solve_b")
    g.add_edge("solve_b", "debate")
    g.add_conditional_edges("debate", _should_continue, {"solve_a": "solve_a", "judge": "judge"})
    g.add_edge("judge", END)
    compiled = g.compile()

    return compiled.invoke({
        "topic": topic, "position_a": "", "position_b": "",
        "critique_a": "", "critique_b": "", "round": 0, "verdict": "",
    })


test_topic = "Should software teams adopt trunk-based development over feature branching?"

print(f"Topic: {test_topic}")
print()

for rounds in [1, 2]:
    r = run_debate(test_topic, rounds)
    pa_len = len(r["position_a"].split())
    pb_len = len(r["position_b"].split())
    import re
    verdict_lines = [l for l in r["verdict"].split("\n") if "WINNER" in l.upper()]
    winner = verdict_lines[0].strip() if verdict_lines else "(no WINNER tag found)"
    print(f"MAX_ROUNDS={rounds}:")
    print(f"  Position A: {pa_len} words")
    print(f"  Position B: {pb_len} words")
    print(f"  {winner}")
    print()

In [ ]:
# ─── 6-B: Experiment with a structured scoring judge ──────────────────────────
# The judge prompt shapes how structured the verdict is.
# Here we try a judge that rates each position on three explicit criteria.

DETAILED_JUDGE_SYSTEM = (
    "You are an impartial judge evaluating a structured debate. "
    "Score each position on three criteria (1-5 each):\n"
    "  1. Evidence quality (are claims concrete and verifiable?)\n"
    "  2. Logical coherence (does the argument follow from its premises?)\n"
    "  3. Practical applicability (would this advice work for a real engineer?)\n"
    "Sum the scores and declare the higher scorer the winner. "
    "End with exactly: WINNER: [SPECIALIZATION or GENERALIZATION]"
)


def judge_detailed(state: DebateState) -> dict:
    content = (
        f"Topic: {state['topic']}\n\n"
        f"POSITION A (Specialization):\n{state['position_a']}\n\n"
        f"POSITION B (Generalization):\n{state['position_b']}"
    )
    result = llm.invoke([
        SystemMessage(content=DETAILED_JUDGE_SYSTEM),
        HumanMessage(content=content),
    ])
    return {"verdict": result.content}


# Re-use the existing positions from Part 5 — no extra agent calls needed
detailed_verdict = judge_detailed(result)
print("Detailed scoring judge verdict:")
print(detailed_verdict["verdict"])

---

## Part 7 — Parsing Verdicts and Tracking Wins

---

The `WINNER:` tag at the end of the judge's verdict is a deliberate design choice: it makes the output machine-parseable without an additional LLM call.

This is the **structured output via suffix** pattern — cheaper and more reliable than asking the LLM to return JSON when the rest of the response is free-form prose.

### Judge Design Patterns

| Pattern | How it works | Trade-off |
|---------|-------------|----------|
| **Suffix tag** (this notebook) | `WINNER: [X]` at end of prose | Simple; works with any model |
| **JSON mode** | Force entire output to JSON | Structured; loses prose explanation |
| **Tool call** | Judge calls a `submit_verdict` tool | Clean types; requires function-calling support |
| **Majority vote** | Run N judges; take the mode | Robust to outliers; expensive |

In [ ]:
# ─── 7-A: Parse the WINNER tag with a regex ───────────────────────────────────
import re


def parse_winner(verdict: str) -> str:
    """Extract the WINNER value from a judge verdict string."""
    match = re.search(r"WINNER:\s*(\w+)", verdict, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return "UNPARSEABLE"


winner = parse_winner(result["verdict"])
print(f"Parsed winner from Part 5 result: {winner}")
print()

# Robustness tests — verify the parser handles edge cases
edge_cases = [
    ("...strong argument. WINNER: SPECIALIZATION", "SPECIALIZATION"),
    ("WINNER: generalization", "GENERALIZATION"),
    ("Winner: Specialization\n", "SPECIALIZATION"),
    ("No winner tag here", "UNPARSEABLE"),
]

print("Parser edge case tests:")
for text, expected in edge_cases:
    got = parse_winner(text)
    status = "PASS" if got == expected else f"FAIL (got {got})"
    print(f"  [{status}]  input='{text[:45]}'  ->  {got}")

In [ ]:
# ─── 7-B: Run multiple topics and track win rates ─────────────────────────────
# Run the debate on several topics to see if one side wins disproportionately.
# A consistent bias suggests the judge prompt or system prompts need balancing.

topics = [
    "Is specialization or generalization better for a software engineer's career?",
    "Should software teams adopt trunk-based development over feature branching?",
    "Is remote work better than in-office work for software engineering teams?",
]

wins: dict = {}

print("Multi-topic win rate experiment (1 round each to limit cost)")
print()

for topic in topics:
    r = run_debate(topic, max_rounds=1)
    w = parse_winner(r["verdict"])
    wins[w] = wins.get(w, 0) + 1
    print(f"  Topic: {topic[:65]}")
    print(f"  Winner: {w}")
    print()

print("Win totals:")
for side, count in sorted(wins.items(), key=lambda x: -x[1]):
    bar = "#" * count
    print(f"  {side:22} {bar:10} ({count}/{len(topics)})")

---

## Exercises

---

Work through these in order. Each builds on the one before.

In [ ]:
# ── Exercise 1 — Change the topic ─────────────────────────────────────────────
# Replace MY_TOPIC with any yes/no proposition and re-run the graph.
# Good topic types: technical trade-offs, process decisions, architecture choices.
#
# Observe: does debate quality change with the topic domain?
# Does the judge behave differently on more contentious questions?

# TODO: change this to any proposition you want to debate
MY_TOPIC = "Should AI replace human code review in software development?"

# TODO: run the debate and inspect the verdict
# my_result = run_debate(MY_TOPIC, max_rounds=2)

# TODO: print position_a, position_b, and verdict
# TODO: parse the winner with parse_winner(my_result["verdict"])

print(f"Topic set to: {MY_TOPIC}")
print("Uncomment the lines above and run to see the debate.")

In [ ]:
# ── Exercise 2 — Add a third debater ──────────────────────────────────────────
# Add SOLVER_C_SYSTEM arguing a MIDDLE GROUND or HYBRID position.
# Add position_c and critique_c to a new ThreeWayDebateState TypedDict.
# Add a solve_c node and wire it in after solve_b.
# Update the judge prompt to handle three positions and a HYBRID option.
#
# Challenge: update parse_winner() to accept HYBRID as a valid outcome.

# TODO: define SOLVER_C_SYSTEM
SOLVER_C_SYSTEM = (
    # Suggested: argue for a HYBRID path — specialization first 3-5 years, then broaden
    "TODO: define a middle-ground position"
)

# TODO: define ThreeWayDebateState with position_c and critique_c fields

# TODO: define solve_c node (mirror of solve_a with SOLVER_C_SYSTEM)

# TODO: wire graph: solve_a -> solve_b -> solve_c -> debate

# TODO: update debate node to produce critique_c

# TODO: update judge node to read all three positions

print("Three-way debate skeleton ready. Fill in the TODOs above.")

In [ ]:
# ── Exercise 3 — Temperature experiment ───────────────────────────────────────
# Run the same topic at temperature=0 and temperature=1.
# temperature=0: deterministic; positions may converge faster.
# temperature=1: creative; may introduce unexpected framings.
#
# Compare: which temperature produces the more interesting debate?
# Which produces a more consistent judge verdict across runs?

TEMPERATURE_TOPIC = "Is microservices or monolith architecture better for early-stage startups?"

# TODO: create two ChatOpenAI instances:
#   llm_cold = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
#   llm_hot  = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

# TODO: create node functions that use each llm (pass llm as a parameter
#       rather than relying on the module-level global)

# TODO: run the same topic with both and compare:
#   - Position word count
#   - Uniqueness of arguments (do both use the same examples?)
#   - Whether the judge changes its verdict between the two runs

print(f"Topic: {TEMPERATURE_TOPIC}")
print("TODO: implement the temperature comparison above.")

In [ ]:
# ── Exercise 4 — Consensus mode (no judge) ────────────────────────────────────
# Some debate frameworks stop when both agents agree, measured by semantic
# similarity, rather than using a fixed round count and a judge.
#
# Implement a convergence_check routing function that:
#   1. Embeds position_a and position_b with OpenAIEmbeddings
#   2. Computes cosine similarity
#   3. Returns "done" if similarity > 0.85, else "solve_a"
#   4. Has a safety valve: if round > 4, stop regardless
#
# This replaces the round-count stopping condition.

# TODO: from langchain_openai import OpenAIEmbeddings
# TODO: embed = OpenAIEmbeddings(model="text-embedding-3-small")

# TODO: implement cosine_similarity(a, b) using math.sqrt + zip
import math

def cosine_similarity(a: list, b: list) -> float:
    # TODO: implement dot product, magnitudes, return dot/(mag_a*mag_b)
    pass

# TODO: implement convergence_check(state) that returns "done" or "solve_a"

# TODO: replace add_conditional_edges to use convergence_check

print("Consensus mode skeleton ready. Implement cosine_similarity and convergence_check.")

---

## Part 8 ★ — Extensions (Bonus)

---

### Streaming Debate Output

LangGraph supports streaming at the node level. Instead of waiting for the full result, emit each node's output as it completes — useful for UI integrations:

```python
for event in app.stream(initial_state, stream_mode="values"):
    # event is the full state dict after each node completes
    if event.get("position_a"):
        print(f"[Agent A] {event['position_a'][:80]}...")
```

### Parallel Critique Generation

The `debate` node currently makes two LLM calls sequentially. Parallelize them with `asyncio` to halve latency:

```python
import asyncio

async def debate_async(state: DebateState) -> dict:
    crit_a_task = llm.ainvoke([SystemMessage(SOLVER_A_SYSTEM), HumanMessage(crit_a_content)])
    crit_b_task = llm.ainvoke([SystemMessage(SOLVER_B_SYSTEM), HumanMessage(crit_b_content)])
    critique_a, critique_b = await asyncio.gather(crit_a_task, crit_b_task)
    return {"critique_a": critique_a.content, "critique_b": critique_b.content, "round": ...}
```

### Majority-Vote Judge

Run 3 or 5 independent judge calls and take the majority vote — eliminates single-judge variance:

```python
from collections import Counter

def judge_majority(state, n=3):
    verdicts = [judge(state)["verdict"] for _ in range(n)]
    winners = [parse_winner(v) for v in verdicts]
    winner = Counter(winners).most_common(1)[0][0]
    return {"verdict": f"Majority ({winner}): {winners}"}
```

In [ ]:
# ─── 8-A: Stream the debate output node by node ───────────────────────────────
# stream_mode="values" emits the full state dict after each node completes.
# Print progress as it arrives rather than waiting for the whole run.

stream_topic = "Is Python or Go the better choice for building CLI tools?"

stream_app = create_debate_graph()
stream_initial: DebateState = {
    "topic": stream_topic,
    "position_a": "", "position_b": "",
    "critique_a": "", "critique_b": "",
    "round": 0, "verdict": "",
}

print(f"Streaming debate: {stream_topic}")
print("=" * 60)

prev_pa, prev_pb = "", ""

for event in stream_app.stream(stream_initial, stream_mode="values"):
    if event.get("position_a") and event["position_a"] != prev_pa:
        print(f"\n[Agent A — after round {event.get('round', 0)}]")
        print(event["position_a"][:200] + "...")
        prev_pa = event["position_a"]
    if event.get("position_b") and event["position_b"] != prev_pb:
        print(f"\n[Agent B — after round {event.get('round', 0)}]")
        print(event["position_b"][:200] + "...")
        prev_pb = event["position_b"]
    if event.get("verdict"):
        print(f"\n[Judge]")
        print(event["verdict"])

In [ ]:
# ─── 8-B: Majority-vote judge ─────────────────────────────────────────────────
# Run the same judge prompt N times on the same positions and take the majority.
# Eliminates single-judge variance for close debates.

from collections import Counter


def judge_majority_vote(state: DebateState, n_judges: int = 3) -> dict:
    """Run n_judges independent judge calls; return majority verdict."""
    verdicts = [judge(state)["verdict"] for _ in range(n_judges)]
    winners = [parse_winner(v) for v in verdicts]
    most_common = Counter(winners).most_common(1)[0][0]
    tally = dict(Counter(winners))

    summary = f"Majority: {most_common}  (vote tally: {tally})\n\nAll verdicts:\n"
    for i, v in enumerate(verdicts, 1):
        first_line = v.split("\n")[0][:80]
        summary += f"  Judge {i}: {first_line}\n"

    return {"verdict": summary}


# Re-use the existing debate result from Part 5
majority_result = judge_majority_vote(result, n_judges=3)
print("Majority-vote verdict (3 independent judges):")
print(majority_result["verdict"])

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

---

### Exercise 1 — Change the Topic

**Sample answer:** Any yes/no or A-vs-B proposition works. Debate quality is highest when both positions have real-world evidence to draw on — technical architecture choices, engineering process decisions, and hiring philosophy all work well.

**What good output looks like:**
- Each agent's position is 3-5 sentences with at least one concrete example or statistic
- Final positions differ meaningfully from opening arguments (sign of productive refinement)
- The judge's reasoning explicitly names which argument was stronger and why

**If both positions sound identical:** the system prompts need stronger locking language. Add: "You must argue exclusively for X — do not concede the overall point under any circumstances."

In [ ]:
# Sample answer for Exercise 1
SAMPLE_TOPIC = "Should AI replace human code review in software development?"

sample_result = run_debate(SAMPLE_TOPIC, max_rounds=2)

print(f"Topic: {SAMPLE_TOPIC}")
print("=" * 60)
print("Position A (For AI-driven review):")
print(sample_result["position_a"][:300] + "...")
print()
print("Position B (For human review):")
print(sample_result["position_b"][:300] + "...")
print()
print("Winner:", parse_winner(sample_result["verdict"]))

### Exercise 2 — Three-Way Debate

**Key implementation notes:**
- Add `position_c: str` and `critique_c: str` to the TypedDict
- `solve_c` is a mirror of `solve_a` with `SOLVER_C_SYSTEM` and `position_c`/`critique_c` keys
- The `debate` node needs to produce critiques for all three pairs — or just the direct opponents depending on your design
- The judge prompt must list three positions and `parse_winner` must accept a third value

**Common mistake:** Mixing up which agent receives which critique. Name the fields clearly: `critique_received_by_a`, `critique_received_by_b`, `critique_received_by_c`.

In [ ]:
# Sample answer skeleton for Exercise 2
from typing import TypedDict as TD


class ThreeWayDebateState(TD):
    topic: str
    position_a: str
    position_b: str
    position_c: str       # <- new: Hybrid advocate
    critique_a: str       # what A must respond to
    critique_b: str       # what B must respond to
    critique_c: str       # what C must respond to
    round: int
    verdict: str


SOLVER_C_SYSTEM_SAMPLE = (
    "You are a career expert arguing for a HYBRID approach: engineers should "
    "build a deep specialty first (3-5 years) then deliberately broaden to T-shaped. "
    "Make compelling, evidence-based arguments in 3-4 sentences. "
    "If given a critique, acknowledge its strongest point then reinforce your position."
)

JUDGE_THREE_WAY_SYSTEM = (
    "You are an impartial judge evaluating a three-way debate. "
    "Read all three positions and select the strongest argument. "
    "Explain your reasoning in 2-3 sentences, then end with exactly: "
    "WINNER: [SPECIALIZATION or GENERALIZATION or HYBRID]"
)

print("ThreeWayDebateState defined with position_c and critique_c fields.")
print("Wire solve_c after solve_b, update debate to produce critique_c.")

### Exercise 3 — Temperature Experiment

**Expected findings:**
- `temperature=0`: positions are more concise and formulaic; critiques may be generic; judge tends to be consistent across multiple runs on the same topic
- `temperature=1`: positions introduce unexpected framings or analogies; critiques are more creative but occasionally off-topic; judge verdicts vary more across runs

**Practical recommendation:** `temperature=0.7` for agents (diverse but coherent), `temperature=0` for the judge (consistent evaluation). This mirrors the pattern used in LLM-as-judge evaluation harnesses.

In [ ]:
# Sample answer for Exercise 3 — temperature comparison

temp_topic = "Is microservices or monolith architecture better for early-stage startups?"
print(f"Temperature experiment: 0.0 vs 1.0")
print(f"Topic: {temp_topic}")
print()

for temp in [0.0, 1.0]:
    temp_llm = ChatOpenAI(model="gpt-4o-mini", temperature=temp)
    opening_a = temp_llm.invoke([
        SystemMessage(content=SOLVER_A_SYSTEM),
        HumanMessage(content=f"Topic: {temp_topic}\n\nPresent your opening argument FOR SPECIALIZATION:"),
    ])
    opening_b = temp_llm.invoke([
        SystemMessage(content=SOLVER_B_SYSTEM),
        HumanMessage(content=f"Topic: {temp_topic}\n\nPresent your opening argument FOR GENERALIZATION:"),
    ])
    word_a = len(opening_a.content.split())
    word_b = len(opening_b.content.split())
    print(f"temperature={temp}:")
    print(f"  Agent A ({word_a} words): {opening_a.content[:120]}...")
    print(f"  Agent B ({word_b} words): {opening_b.content[:120]}...")
    print()

### Exercise 4 — Consensus Mode

**Key implementation:**
- Use `OpenAIEmbeddings` to embed `position_a` and `position_b`
- Compute cosine similarity (same formula as the RAG notebook)
- Replace `should_continue` with `convergence_check` in `add_conditional_edges`
- Always add a safety valve (`round > MAX_ROUNDS_SAFETY`) to prevent infinite loops

**Expected behavior:** On clearly controversial topics, similarity rarely exceeds 0.85 even after many rounds — agents hold their positions. On topics with a factual answer, similarity converges faster and the debate terminates early.

In [ ]:
# Sample answer for Exercise 4 — convergence-based stopping
import math


def cosine_similarity_impl(a: list, b: list) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0


def make_convergence_check(embed_model, threshold: float = 0.85, max_rounds_safety: int = 4):
    """Return a routing function that stops when positions converge semantically."""
    def convergence_check(state: DebateState) -> str:
        if state.get("round", 0) >= max_rounds_safety:
            return "judge"  # safety valve — always terminate eventually
        vecs = embed_model.embed_documents([state["position_a"], state["position_b"]])
        sim = cosine_similarity_impl(vecs[0], vecs[1])
        print(f"  [convergence] round={state['round']}, cosine_similarity={sim:.3f}")
        return "judge" if sim >= threshold else "solve_a"
    return convergence_check


print("Usage:")
print("  from langchain_openai import OpenAIEmbeddings")
print("  embed = OpenAIEmbeddings(model='text-embedding-3-small')")
print("  check = make_convergence_check(embed, threshold=0.85)")
print("  graph.add_conditional_edges('debate', check, {'solve_a': 'solve_a', 'judge': 'judge'})")

---

## What's Next?

You now have a complete foundation in multi-agent debate with LangGraph. Here is where to go deeper:

### Related examples in this repo
- **Example 5 — ReAct Agent with LangGraph** (`../5-react-agent-lg/`): two-agent critic loop — a different pattern where one agent summarizes and another reviews
- **Example 18 — Self-Reflecting Agent** (`../18-self-reflecting-agent/`): single-agent self-critique — the cheaper sibling of debate for cost-sensitive pipelines
- **Example 30 — Agentic RAG** (`../30-agentic-rag/`): combine debate with RAG — agents argue from retrieved evidence rather than training knowledge alone

### Improve debate quality
- **Steelmanning prompts**: instruct each agent to *first* state the strongest version of the opponent's argument before critiquing — forces more rigorous reasoning
- **Role differentiation**: give agents different personas (practitioner vs researcher vs contrarian) rather than just position labels
- **Evidence injection**: attach retrieved documents to each agent's context so positions are grounded in real sources

### Production patterns
- **LangSmith tracing**: add `LANGCHAIN_TRACING_V2=true` to your `.env` to see the full graph execution in a visual dashboard
- **Async nodes**: convert all LLM calls to `ainvoke` for non-blocking execution in FastAPI or other async frameworks
- **Verdict database**: store `(topic, winner, position_a, position_b, verdict)` rows to track judge consistency and detect prompt drift over time

### Further reading
- Du, Y. et al. (2023). *Improving Factuality and Reasoning in Language Models through Multiagent Debate.* ICML 2024. https://arxiv.org/abs/2305.14325
- Liang, T. et al. (2023). *Encouraging Divergent Thinking in Large Language Models through Multi-Agent Debate.* https://arxiv.org/abs/2305.19118
- Chan, C.-M. et al. (2023). *ChatEval: Towards Better LLM-based Evaluators through Multi-Agent Debate.* https://arxiv.org/abs/2308.07201
- LangGraph concepts — nodes, edges, state, streaming: https://langchain-ai.github.io/langgraph/concepts/

---

*Workshop complete. The next step is Example 18 — see how a single agent critiques itself without an opponent.*